### 9. Qual é a taxa de recursos (Agravos de Instrumento) interpostos contra decisões que deferem ou indeferem o processamento da RJ e qual a taxa de reforma dessas decisões no 2º Grau?

**Resposta sintética:** a métrica não é computável com os dados disponíveis pelo Datajud do CNJ e o diagnóstico do porquê é o achado deste notebook.

**O que foi testado:** vincular agravos de instrumento (classe TPU 202, 30.750 processos no corpus) às RJs (classe 129, 5.709 processos após 09/06/2005) pela chave raiz+origem do número CNJ — o mesmo método validado na pergunta 7 para incidentes da mesma vara (tutela cautelar antecedente).

**Resultado: 0 vínculos em 30.750 agravos.** O motivo é estrutural: a chave raiz+origem do número CNJ identifica processos que nasceram na mesma vara (foi assim que vinculamos a tutela antecedente às RJs na pergunta 8). O agravo de instrumento, porém, é um recurso autuado no
tribunal, com numeração própria — raiz sequencial do 2º grau e unidade de origem do tribunal. Processo e recurso, portanto, nunca compartilham a chave: o vínculo é impossível com o dado disponível. E sem saber a qual RJ cada agravo pertence, não há sobre quais autos
medir o desfecho de 2º grau.

Nem a delimitação pelo assunto-pai 4993 (recuperação judicial e falências) resolve: no corpus, os agravos de instrumento orbitam classes originárias diversas desse assunto — 108 (Falência), 128 (Recuperação Extrajudicial), 129 (Recuperação Judicial), entre outras —, mas o número CNJ do recurso não preserva nem a classe nem a vara do
processo originário, de modo que não há como atribuir um agravo a qualquer uma dessas classes.

**Por que isso rjustifica a não resposta à pergunta:** o Datajud não registra o vínculo recurso→decisão agravada, e a numeração CNJ dos recursos não preserva a origem — a taxa de agravabilidade e a taxa de reforma são estruturalmente não deriváveis do acervo. É o segundo não-achado estrutural do projeto (o primeiro: a homologação do plano, pergunta 3).

**Universo (filtros):** RJs classe 129 com escopo_lei_atual; agravos classe 202 do corpus completo.

#### O que a próxima consulta de QA verifica

**Entrada:** a tabela `default.gold_base_analitica` (somente colunas de
número de processo e classe), de onde a consulta extrai, para dois
universos — RJs (classe 129, universo da lei) e agravos de instrumento
(classe 202) — apenas os dois componentes da chave CNJ: a **raiz**
(7 primeiros dígitos) e a **origem** (4 últimos). Todo o resto do
registro é descartado: a pergunta é só sobre a chave.

**Por que existe:** a consulta principal de vínculo exigiu raiz **e**
origem iguais entre agravo e RJ — e devolveu 0 em 30.750 agravos. Antes
de aceitar o zero como resultado, é preciso descartar a hipótese de bug:
a chave pode ter falhado por um componente isolado (ex.: origens batem,
raízes não). Este QA decompõe o vínculo e testa cada componente
isoladamente:

- `agravos_mesma_raiz` — quantos agravos têm raiz igual à de alguma RJ,
  sem exigir origem (se a raiz do agravo nunca coincide, é numeração
  própria do tribunal);
- `agravos_origem_de_vara_rj` — quantos agravos têm origem igual à de
  alguma RJ (se ~0, o agravo não nasce em vara de 1º grau);
- `origens_distintas_agravos` × `origens_distintas_rj` — o contraste
  entre os conjuntos de unidades de origem dos dois universos: se os
  agravos moram em poucas unidades (48) e as RJs em muitas (397), são
  dois mundos de numeração, não o mesmo.

**Saída (uma linha com cinco números):** o tamanho do universo de
agravos, as duas contagens de coincidência por componente isolado e os
tamanhos dos dois conjuntos de unidades de origem.

**Resultado esperado conforme o diagnóstico:**
- coincidências por componente ≈ 0 e conjuntos de origem incompatíveis
  (48 × 397) → o zero da consulta principal é **estrutura do dado**, não
  bug: o agravo é autuado no tribunal com numeração própria e jamais
  compartilha a chave com a RJ — o vínculo é impossível;
- coincidência alta em um componente → haveria caminho alternativo de
  vínculo a explorar (ex.: origem + janela temporal) antes de fechar a
  pergunta.

#### Como funciona o próximo código

**Passo 1 — duas CTEs (blocos temporários):**

- `rj` — seleciona as RJs do universo da lei (`classe_codigo = 129` +
  `escopo_lei_atual`) e, de cada número de processo, extrai apenas os
  dois componentes da chave CNJ: a **raiz** (`substring(..., 1, 7)` —
  os 7 primeiros dígitos, sequência da vara) e a **origem**
  (`substring(..., 17, 4)` — os 4 últimos, código da unidade
  judiciária). Guarda também `data_ajuizamento` e
  `data_processamento_efetivo`, necessárias para as janelas;
- `agravos` — faz o mesmo para os agravos de instrumento
  (`classe_codigo = 202`), guardando a data de ajuizamento do agravo.

O objetivo das CTEs é reduzir cada processo a um par (raiz, origem)
mais as datas: só o que a análise usa.

**Passo 2 — o JOIN (a tentativa de vínculo):**

`FROM agravos a JOIN rj r ON r.raiz = a.raiz AND r.origem = a.origem`
exige que agravo e RJ tenham **raiz e origem iguais ao mesmo tempo** —
a chave que identificou processos irmãos da mesma vara na pergunta 8.
Se nenhum agravo casar, o JOIN devolve zero linhas.

**Passo 3 — as sete medidas da saída (uma linha só):**

- `total_rj`, `rj_com_processamento`, `rj_sem_processamento` — os
  denominadores das taxas (5.709 RJs; 2.422 com e 3.287 sem
  processamento datado), contados por subconsultas sobre a CTE `rj`
  (por isso sobrevivem mesmo quando o JOIN não casa);
- `total_agravos_202` — o universo de agravos (30.750), também por
  subconsulta independente do JOIN;
- `agravos_vinculados_a_rj` — **a medida central**: conta as linhas
  que sobreviveram ao JOIN, ou seja, quantos agravos se vincularam a
  alguma RJ;
- `janela_a_agravos` — dos vinculados, os ajuizados entre o
  ajuizamento da RJ e 90 dias após o processamento efetivo
  (`CASE WHEN` com as três condições: posterior ao ajuizamento,
  processamento existente, e dentro da janela de 90 dias);
- `janela_b_agravos` — dos vinculados, os ajuizados após o ajuizamento
  em RJs sem processamento datado (proxy do indeferimento).

**Detalhe técnico importante:** os quatro primeiros números usam
subconsultas sobre as CTEs, e os três últimos usam `count` sobre o
resultado do JOIN. É por isso que o output pode mostrar universos
populosos (5.709 RJs, 30.750 agravos) e, ao mesmo tempo, zero nas
colunas de vínculo — sem isso, um JOIN vazio zeraria tudo e esconderia
os denominadores.


In [0]:
%sql
-- Q9 · JANELAS A/B (camada gold): agravos de instrumento (classe 202)
-- vinculados a RJ (classe 129) pela chave raiz+origem CNJ, atribuidos a
-- decisao de processamento por janela temporal. Consome apenas a gold.
WITH rj AS (
  SELECT substring(numero_processo, 1, 7)  AS raiz,
         substring(numero_processo, 17, 4) AS origem,
         data_ajuizamento                  AS data_ajuizamento_rj,
         data_processamento_efetivo
  FROM default.gold_base_analitica
  WHERE classe_codigo = 129 AND escopo_lei_atual
),
agravos AS (
  SELECT numero_processo,
         substring(numero_processo, 1, 7)  AS raiz,
         substring(numero_processo, 17, 4) AS origem,
         data_ajuizamento                  AS data_ajuizamento_agravo
  FROM default.gold_base_analitica
  WHERE classe_codigo = 202
)
SELECT
  (SELECT count(*) FROM rj)                                              AS total_rj,
  (SELECT count(*) FROM rj WHERE data_processamento_efetivo IS NOT NULL) AS rj_com_processamento,
  (SELECT count(*) FROM rj WHERE data_processamento_efetivo IS NULL)     AS rj_sem_processamento,
  (SELECT count(*) FROM agravos)                                         AS total_agravos_202,
  count(a.numero_processo)                                               AS agravos_vinculados_a_rj,
  count(CASE WHEN a.data_ajuizamento_agravo >= r.data_ajuizamento_rj
              AND r.data_processamento_efetivo IS NOT NULL
              AND a.data_ajuizamento_agravo
                    <= date_add(r.data_processamento_efetivo, 90)
             THEN 1 END)                                                 AS janela_a_agravos,
  count(CASE WHEN a.data_ajuizamento_agravo > r.data_ajuizamento_rj
              AND r.data_processamento_efetivo IS NULL
             THEN 1 END)                                                 AS janela_b_agravos
FROM agravos a
JOIN rj r ON r.raiz = a.raiz AND r.origem = a.origem;

#### Análise do resultado da vinculação

**O que o output mostra:**

| Coluna | Valor | Leitura |
|---|---|---|
| `total_rj` | 5.709 | o universo de RJs da lei está íntegro — igual às perguntas 1 e 8 |
| `rj_com_processamento` | 2.422 | denominador da taxa da janela A (contra o deferimento) |
| `rj_sem_processamento` | 3.287 | denominador da taxa da janela B (proxy do indeferimento) |
| `total_agravos_202` | 30.750 | o universo de agravos de instrumento do corpus |
| `agravos_vinculados_a_rj` | **0** | **a medida central: nenhum agravo casou com alguma RJ na chave raiz+origem** |
| `janela_a_agravos` | 0 | consequência direta: sem vínculos, nenhum agravo cai na janela A |
| `janela_b_agravos` | 0 | idem para a janela B |

**A leitura do resultado:** com `agravos_vinculados_a_rj = 0`, as taxas
das janelas são incomputáveis — `0 / 2.422` e `0 / 3.287` não são taxas,
são a ausência de objeto. Mas um zero em 30.750 agravos precisa ser
explicado antes de aceito: pode ser bug de código (chave mal extraída,
JOIN mal condicionado) ou estrutura do dado. **A célula de QA seguinte
faz esse diagnóstico**: decompõe a chave em raiz e origem, testa cada
componente isoladamente e compara os conjuntos de unidades de origem
dos dois universos — para determinar se o zero é artefato ou estrutura.

#### O que a consulta de QA a seguir verifica

**Entrada:** a tabela `default.gold_base_analitica` (somente colunas de
número de processo e classe), de onde a consulta extrai, para dois
universos — RJs (classe 129, universo da lei) e agravos de instrumento
(classe 202) — apenas os dois componentes da chave CNJ: a **raiz**
(7 primeiros dígitos) e a **origem** (4 últimos). Todo o resto do
registro é descartado: a pergunta é só sobre a chave.

**Por que existe:** a consulta principal de vínculo exigiu raiz **e**
origem iguais entre agravo e RJ — e devolveu 0 em 30.750 agravos. Antes
de aceitar o zero como resultado, é preciso descartar a hipótese de bug:
a chave pode ter falhado por um componente isolado (ex.: origens batem,
raízes não). Este QA decompõe o vínculo e testa cada componente
isoladamente:

- `agravos_mesma_raiz` — quantos agravos têm raiz igual à de alguma RJ,
  sem exigir origem (se a raiz do agravo nunca coincide, é numeração
  própria do tribunal);
- `agravos_origem_de_vara_rj` — quantos agravos têm origem igual à de
  alguma RJ (se ~0, o agravo não nasce em vara de 1º grau);
- `origens_distintas_agravos` × `origens_distintas_rj` — o contraste
  entre os conjuntos de unidades de origem dos dois universos: se os
  agravos moram em poucas unidades (48) e as RJs em muitas (397), são
  dois mundos de numeração, não o mesmo.

**Saída (uma linha com cinco números):** o tamanho do universo de
agravos, as duas contagens de coincidência por componente isolado e os
tamanhos dos dois conjuntos de unidades de origem.

**Resultado esperado conforme o diagnóstico:**
- coincidências por componente ≈ 0 e conjuntos de origem incompatíveis
  (48 × 397) → o zero da consulta principal é **estrutura do dado**, não
  bug: o agravo é autuado no tribunal com numeração própria e jamais
  compartilha a chave com a RJ — o vínculo é impossível;
- coincidência alta em um componente → haveria caminho alternativo de
  vínculo a explorar (ex.: origem + janela temporal) antes de fechar a
  pergunta.

In [0]:
%sql
-- Q9 · QA: por que a vinculacao raiz+origem nao capturou NENHUM agravo?
-- Hipotese estrutural: a tutela antecedente (Q7/Q8) e incidente da mesma
-- vara (compartilha raiz+origem); o agravo de instrumento e autuado no
-- tribunal com numero proprio (raiz e origem proprias). Diagnostico
-- separa os dois componentes da chave e compara os universos de origem.
WITH rj AS (
  SELECT substring(numero_processo, 1, 7)  AS raiz,
         substring(numero_processo, 17, 4) AS origem
  FROM default.gold_base_analitica
  WHERE classe_codigo = 129 AND escopo_lei_atual
),
agravos AS (
  SELECT substring(numero_processo, 1, 7)  AS raiz,
         substring(numero_processo, 17, 4) AS origem
  FROM default.gold_base_analitica
  WHERE classe_codigo = 202
)
SELECT
  (SELECT count(*) FROM agravos)                                  AS total_agravos,
  count(CASE WHEN EXISTS (SELECT 1 FROM rj r WHERE r.raiz = a.raiz)
             THEN 1 END)                                          AS agravos_mesma_raiz,
  count(CASE WHEN EXISTS (SELECT 1 FROM rj r WHERE r.origem = a.origem)
             THEN 1 END)                                          AS agravos_origem_de_vara_rj,
  count(DISTINCT a.origem)                                        AS origens_distintas_agravos,
  (SELECT count(DISTINCT origem) FROM rj)                         AS origens_distintas_rj
FROM agravos a;

#### Análise do resultado do QA

**O que os cinco números dizem:**

| Valor | Leitura |
|---|---|
| `total_agravos` = 30.750 | o universo de agravos de instrumento do corpus está íntegro — o zero da vinculação não decorre de universo vazio ou filtro mal aplicado |
| `agravos_mesma_raiz` = 187 (0,6%) | apenas 187 agravos coincidem de raiz com alguma RJ — e mesmo esses **não constituem vínculo**: são colisões estatísticas do espaço de 7 dígitos, pois **nenhum** deles casa também na origem (casaram sempre isolados, nunca os dois componentes juntos — por isso o vínculo completo deu 0) |
| `agravos_origem_de_vara_rj` = 156 (0,5%) | apenas 156 agravos têm alguma origem que também aparece entre as varas das RJs — coincidências de código, sem vínculo demonstrável |
| `origens_distintas_agravos` = 48 | os agravos moram em apenas **48 unidades de origem** distintas — conjunto pequeno, típico de 2º grau (câmaras/gabinetes de tribunal), e não das centenas de varas de 1º grau |
| `origens_distintas_rj` = 397 | as RJs nascem em **397 varas** distintas — o universo de unidades de 1º grau |

**O contraste decisivo: 48 × 397.** Se o agravo fosse registrado na vara da RJ
(como a tutela antecedente da pergunta 8, que compartilha raiz e origem por
nascer na mesma vara), as unidades de origem dos agravos deveriam ser um
subconjunto das 397 varas — e os agravos deveriam casar na origem em volume.
O que se observa é o inverso: os 30.750 agravos concentram-se em 48 unidades
(pequeno conjunto de 2º grau), com apenas 0,5% de sobreposição acidental com
as varas. A amostra comparada (QA 2) confirma o padrão: raízes de agravo
sequenciais baixas (0000001, 0000011, 0000144...) e origens do tipo 0000 e
9xxx — numeração própria do tribunal, sem rastro da vara de origem.

**Conclusão do QA:** o zero da consulta principal é **estrutura do dado**, não
bug de código. A chave raiz+origem identifica processos que nasceram na mesma
vara; o agravo de instrumento é autuado no tribunal com numeração própria e
jamais compartilha a chave com a RJ. O vínculo agravo→processo originário é
impossível com o dado disponível — e, sem saber a qual RJ cada agravo
pertence, não há sobre quais autos medir o desfecho de 2º grau (a taxa de
reforma herda a mesma inviabilidade). Este resultado fundamenta a resposta
metodológica da pergunta 9.

#### Resposta à pergunta 9

**Parte 1 — taxas de recursos (Agravos de Instrumento) interpostos contra
decisões de deferimento/indeferimento do processamento da RJ:** não
computável. O teste de vinculabilidade pela chave raiz+origem do número
CNJ — o método validado na pergunta 8 para incidentes da mesma vara —
retornou 0 vínculos em 30.750 agravos. O QA estrutural demonstrou o
motivo: o agravo é autuado no tribunal com numeração própria (48
unidades de origem do 2º grau, contra as 397 varas de origem das RJs),
de modo que processo e recurso nunca compartilham a chave. Nem a
delimitação pelo assunto-pai 4993 resolve, pois o número do recurso não
preserva nem a classe nem a vara do processo originário.

**Parte 2 — taxa de reforma dessas decisões no 2º Grau:** não computável,
pois herda a inviabilidade da parte 1: sem vínculo agravo→RJ, não há
sobre quais autos medir o desfecho.

**Alternativas de superação investigadas e descartadas neste estudo:**
1. **Campo de processos antecessores na fonte bruta** (`numero_processo_prior`
   do padrão CNJ, onde o recurso apontaria o processo de origem) —
   verificado na silver e no raw: **o campo não existe na ingestão**. A via
   estruturada única possível está, portanto, ausente já na fonte, não
   apenas na modelagem deste projeto;
2. **Metadados estruturados dos movimentos** — o perfil de registro dos
   movimentos, documentado na pergunta 2, não contém vocabulário que
   identifique o processo de origem do recurso.

**Caminhos futuros (fora do escopo deste estudo):** a identificação do
vínculo recurso→processo originário reside no documento processual, não
no dado estruturado. A petição do agravo de instrumento deve ser
instruída, obrigatoriamente, com cópias da petição que ensejou a decisão
agravada e da própria decisão agravada (art. 1.017, I, do CPC) — peças
que identificam o processo de origem —, e a falta de peça que comprometa
a admissibilidade sujeita o recurso ao não-conhecimento (art. 1.017,
§ 3º, c/c art. 932, parágrafo único). O despacho de admissibilidade do
relator, por sua vez, tipicamente referencia o processo de origem. Sua
extração, contudo, demandaria a coleta das peças nos portais eletrônicos
dos tribunais (PJe, e-SAJ), configurando um segundo projeto de aquisição
de dados — os autos eletrônicos dispensam a juntada das cópias (art.
1.017, § 5º), o que confirma que o vínculo habita os autos, não os
metadados. Superado o vínculo, a classificação do objeto do agravo
(decisão de processamento vs. tutela vs. decisão interlocutória) viria
da leitura do documento, eliminando também os falsos positivos da
atribuição por janela temporal previstos no desenho original desta
métrica.
